# Mini-Workshop 1: Clusters from Nothing

A colleague recorded 300 neurons over a 5-second trial and ran a standard
clustering analysis to classify them into functional types. The analysis uses
silhouette scoring to objectively choose the number of clusters, runs k-means,
and displays per-cluster heatmaps and mean response profiles.

Run the cells below to reproduce the analysis, then work through the exercise
to evaluate whether the conclusion holds up.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.ndimage import gaussian_filter1d

plt.rcParams['font.family']       = 'sans-serif'
plt.rcParams['font.sans-serif']   = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['pdf.fonttype']      = 42
plt.rcParams['ps.fonttype']       = 42
rng = np.random.default_rng(0)   # for cosmetic row shuffling only

HEAT_CMAP = 'inferno'   # perceptually-uniform colormap for heatmaps

def cluster_palette(n):
    """n distinct qualitative colors, one per cluster."""
    return plt.cm.tab10(np.arange(n) % 10)

def cluster_label(c, k):
    """Human-readable cluster name. For k=3 use early/middle/late."""
    if k == 3:
        return f'cluster {c} ({["early", "middle", "late"][c]})'
    return f'cluster {c}'


In [ ]:
# Load the dataset: a (neurons x time) matrix of activity.
d = np.load('data/activity.npz')
activity = d['activity']      # shape (n_neurons, n_time)
time     = d['time']          # seconds
n_neurons, n_time = activity.shape
print(f'{n_neurons} neurons x {n_time} time bins ({time[0]:.2f}-{time[-1]:.2f} s)')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## The analysis: choosing k with silhouette scoring

Rather than picking the number of clusters by hand, the standard approach is
**silhouette analysis** - scan a range of k values, score each clustering by
how tightly points sit within their own cluster relative to the nearest other
cluster, and keep the k with the best score.

</div>


In [ ]:
# Scan candidate cluster counts k=2 through 8.
# For each k, run k-means and compute the silhouette score.
# Higher silhouette = tighter, more separated clusters.
candidate_ks = range(2, 9)
sil = {kk: silhouette_score(
           activity,
           KMeans(n_clusters=kk, n_init=10, random_state=0).fit_predict(activity))
       for kk in candidate_ks}
best_k = max(sil, key=sil.get)
print(f'Silhouette analysis selects k = {best_k} clusters.')
print(f'Silhouette scores: {", ".join(f"k={k}: {v:.3f}" for k, v in sil.items())}')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## The analysis: k-means clustering and visualization

With k=3 selected, we cluster the neurons, relabel clusters from early to late
by their mean-trace peak time, and plot per-cluster heatmaps and mean responses.

</div>


In [ ]:
k = 3
km     = KMeans(n_clusters=k, n_init=10, random_state=0)
labels = km.fit_predict(activity)   # assign each neuron to a cluster

# Relabel clusters 0/1/2 = early/middle/late by their mean-trace peak time.
# This is cosmetic only - it does not change the clustering.
peak_of_mean = np.array([activity[labels == c].mean(0).argmax() for c in range(k)])
order        = np.argsort(peak_of_mean)
relabel      = np.zeros(k, dtype=int)
relabel[order] = np.arange(k)
labels = relabel[labels]

for c in range(k):
    print(f'{cluster_label(c, k)}: {(labels == c).sum()} neurons')


In [ ]:
colors = cluster_palette(k)
fig    = plt.figure(figsize=(11, 1.6 * k + 0.6))
gs     = fig.add_gridspec(k, 2, width_ratios=[1, 1.1], hspace=0.7, wspace=0.3)

vmin = np.percentile(activity, 2)
vmax = np.percentile(activity, 99)

# Left column: one heatmap per cluster (rows shuffled within cluster for display)
heat_axes = []
for c in range(k):
    ax  = fig.add_subplot(gs[c, 0])
    idx = rng.permutation(np.where(labels == c)[0])
    im  = ax.imshow(activity[idx], aspect='auto', cmap=HEAT_CMAP,
                    vmin=vmin, vmax=vmax,
                    extent=[time[0], time[-1], 0, len(idx)])
    ax.set_title(cluster_label(c, k), color=colors[c], fontweight='bold')
    ax.set_ylabel('neuron')
    if c == k - 1:
        ax.set_xlabel('time (s)')
    heat_axes.append(ax)
plt.colorbar(im, ax=heat_axes, fraction=0.046, pad=0.02).set_label('activity (a.u.)')

# Right column: mean +/- SEM trace per cluster
ax = fig.add_subplot(gs[:, 1])
for c in range(k):
    m   = activity[labels == c].mean(0)
    sem = activity[labels == c].std(0) / np.sqrt((labels == c).sum())
    ax.plot(time, m, color=colors[c], lw=2, label=cluster_label(c, k))
    ax.fill_between(time, m - sem, m + sem, color=colors[c], alpha=0.3)
ax.set_xlabel('time (s)'); ax.set_ylabel('activity (a.u.)')
ax.set_title('Mean response by cluster'); ax.legend(frameon=False)

fig.suptitle('Three temporal response classes?', fontsize=14, fontweight='bold')
plt.show()


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

The silhouette analysis selected k=3. The three clusters show clearly separated
mean traces and each cluster's heatmap appears internally coherent. A colleague
concludes:

> *"The population contains three distinct functional classes of neurons -
> **early**, **middle**, and **late** responders - each with its own reliable
> temporal response profile."*

**The claim is false.** Work through the exercise below to find out why.

</div>


<div style="border-left: 3px solid #07bc0a; padding: 1px; padding-left: 10px; background: #DFF0D8; max-width: 90%; overflow-x: auto; color: #000000;">

### Exercise: Evaluate the cluster analysis

Before accepting the conclusion of three response classes, work through the
following questions. The goal is to identify what the analysis assumes and
what checks it omits.

<ol>

<li><strong>What does k-means guarantee?</strong> Given that you ask for k=3 groups,
will k-means ever return fewer - for example, if the data show no natural
separation? What does this mean for interpreting the output?
<details>
<summary>Hint</summary>

*k-means is required to assign every data point to exactly one of k groups.
It will always return k non-empty clusters regardless of whether any real
grouping exists. Try changing <code>k = 3</code> to <code>k = 4</code> or <code>k = 5</code> in the
clustering cell and re-run - what do you get?*

</details>
</li>

<br>

<li><strong>How large is the silhouette score?</strong> Look at the values printed above.
As a rough rule of thumb, a silhouette below ~0.25 indicates no substantial
cluster structure. What does the peak value here suggest?
<details>
<summary>Hint</summary>

*A silhouette score always names a "best k" - but the value tells you how
real the structure is. A low peak value at k=3 means k=3 is the best of a
set of poor options, not evidence that 3 clusters genuinely exist.*

</details>
</li>

<br>

<li><strong>Design a check.</strong> If there truly were three discrete response classes,
each neuron would belong to one class and its peak activity time would fall
within a narrow range for that class. What would the <em>distribution</em> of
peak times look like across all neurons for three real classes? For a smooth
continuum?

Estimate each neuron's peak time (lightly smooth its trace and take the
argmax), then plot the distribution. Does it look trimodal?
<details>
<summary>Hint</summary>

*Use <code>gaussian_filter1d(activity, sigma=2.0, axis=1)</code> to smooth, then
<code>time[smoothed.argmax(1)]</code> to get each neuron's peak time in seconds.
Plot as a histogram.*

</details>
</li>

<br>

<li><strong>Visualize without presupposing groups.</strong> The per-cluster heatmaps split
neurons into three panels, which invites the eye to see three blocks. Sort
<em>all</em> neurons together by peak time and display them in a single heatmap.
What does the full population look like?
<details>
<summary>Hint</summary>

*Sort the rows of <code>activity</code> by <code>np.argsort(peak_time_est)</code> and display
with <code>imshow</code>. Optionally mark where k-means draws its boundaries.*

</details>
</li>

</ol>

</div>


## Solution


In [ ]:
# ── Evidence 1: Distribution of peak times ────────────────────────────────
# Lightly smooth each neuron's trace, then take the time of its maximum
# as an estimate of that neuron's peak time.
smoothed      = gaussian_filter1d(activity, sigma=2.0, axis=1)
peak_time_est = time[smoothed.argmax(1)]   # peak time (s) for each neuron

fig, ax = plt.subplots(figsize=(6, 3.5))
for c in range(k):
    ax.hist(peak_time_est[labels == c],
            bins=np.linspace(time[0], time[-1], 26),
            color=colors[c], alpha=0.7, label=cluster_label(c, k))
ax.set_xlabel('estimated peak time (s)')
ax.set_ylabel('number of neurons')
ax.set_title('Peak times are continuous and uniform, not trimodal')
ax.legend(frameon=False)
plt.show()

# If there were three response classes, the histogram would have three modes.
# Instead it is flat - a smooth continuum with no gaps.
for c in range(k):
    pt = peak_time_est[labels == c]
    print(f'cluster {c}: peak times span [{pt.min():.2f}, {pt.max():.2f}] s'
          f'  - a contiguous slice, not a natural group')


In [ ]:
# ── Evidence 2: Sort all neurons by peak time in one heatmap ──────────────
# The per-cluster heatmaps used three separate panels and arbitrary row order,
# which invites the eye to see three distinct blocks. Putting all neurons in
# one heatmap, sorted by peak time, shows the truth: a single smooth diagonal.
sort_idx = np.argsort(peak_time_est)
vmin = np.percentile(activity, 2)
vmax = np.percentile(activity, 99)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(activity[sort_idx], aspect='auto', cmap=HEAT_CMAP,
               vmin=vmin, vmax=vmax,
               extent=[time[0], time[-1], 0, n_neurons])
# Mark where k-means draws its boundaries (at cumulative cluster sizes).
sizes = [(labels == c).sum() for c in range(k)]
for b in np.cumsum(sizes)[:-1]:
    ax.axhline(b, color='red', lw=1.5, ls='--')
ax.set_xlabel('time (s)')
ax.set_ylabel('neuron (sorted by peak time)')
ax.set_title('One continuum, cut into three (red = k-means boundaries)')
plt.colorbar(im, ax=ax, label='activity (a.u.)')
plt.show()


In [ ]:
# ── Evidence 3: PCA shows a continuous arch, not separated blobs ──────────
# If three discrete clusters existed we would see three separated clouds in
# PC space. Instead neurons lie on a smooth 1-D arch - the signature of a
# single latent variable (peak time) varying continuously.
Xc = activity - activity.mean(0)       # center each time point
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pcs     = U * S                        # neuron coordinates in PC space
var_exp = S**2 / np.sum(S**2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
for c in range(k):
    m = labels == c
    ax.scatter(pcs[m, 0], pcs[m, 1], s=14, color=colors[c],
               alpha=0.8, label=cluster_label(c, k))
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.0f}% var)')
ax.set_title('Colored by k-means cluster'); ax.legend(frameon=False)

ax = axes[1]
sc = ax.scatter(pcs[:, 0], pcs[:, 1], s=14, c=peak_time_est, cmap='viridis')
ax.set_xlabel(f'PC1 ({var_exp[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({var_exp[1]*100:.0f}% var)')
ax.set_title('Colored by peak time')
plt.colorbar(sc, ax=ax, label='peak time (s)')

fig.suptitle('One continuous arch, not three clusters', fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# ── Evidence 4: The silhouette peak at k=3 is not evidence of clusters ────
# Silhouette always names a "best k". The decisive check: compare the real
# data against no-cluster surrogates - data we KNOW has no groups.
# If the peak at k=3 were evidence of clusters, these surrogates should not
# reproduce it. They do, at the same low value.

ks = list(range(2, 9))

def silhouette_curve(X):
    return [silhouette_score(X,
            KMeans(n_clusters=kk, n_init=10, random_state=0).fit_predict(X))
            for kk in ks]

sil_real = silhouette_curve(activity)

def simulate_continuous(seed, shape=activity.shape):
    """One smooth bump per neuron at a random time - no groups."""
    n, T = shape
    r    = np.random.default_rng(seed)
    peak = r.uniform(0.15 * T, 0.85 * T, size=n)
    t    = np.arange(T)[None, :]
    bumps = 4.0 * np.exp(-0.5 * ((t - peak[:, None]) / 9.0) ** 2)
    return r.normal(0, 1.0, size=(n, T)) + r.uniform(0.7, 1.3, size=(n, 1)) * bumps

sil_surr = np.array([silhouette_curve(simulate_continuous(s)) for s in range(12)])

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.fill_between(ks, sil_surr.min(0), sil_surr.max(0), color='tab:blue',
                alpha=0.25, label='no-cluster surrogates (range)')
ax.plot(ks, sil_surr.mean(0), color='tab:blue', lw=1.5, ls='--',
        label='no-cluster surrogates (mean)')
ax.plot(ks, sil_real, 'o-', color='k', lw=2, label='real data')
ax.axhline(0.25, color='0.5', ls=':', lw=1)
ax.text(8, 0.255, 'weak-structure floor (~0.25)',
        ha='right', va='bottom', color='0.4', fontsize=9)
ax.axvline(3, color='red', ls='--', lw=1)
ax.set_xlabel('number of clusters k'); ax.set_ylabel('silhouette score')
ax.set_title('Real data (k=3 peak) is indistinguishable from no-cluster data')
ax.set_ylim(0, 0.35); ax.legend(frameon=False, fontsize=9)
plt.show()

print(f'real data:  best k = {ks[int(np.argmax(sil_real))]}, '
      f'peak silhouette = {max(sil_real):.3f}')
print(f'surrogates: peak silhouette = {sil_surr.max(1).mean():.3f} '
      f'(mean across 12 datasets we know have no clusters)')


<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; max-width: 90%; overflow-x: auto; color: #000000;">

## Key takeaway

There are no clusters. Every neuron has a single smooth bump peaking at a
randomly chosen, continuous time. K-means with k=3 simply sliced that continuum
into three contiguous bins and called them classes.

- **K-means always returns k clusters.** Asking for 3 groups guarantees 3 groups,
  whether or not any exist. The output is never evidence that discrete classes are present.
- **A "best k" is not evidence of clusters.** Silhouette, gap statistic, and BIC
  select the best k *assuming you cluster at all*. They happily peak on a continuum.
  Always read the value, and compare it against a matched no-cluster null.
- **Averaging within groups hides within-group spread.** The tidy mean traces
  conceal that peak times vary continuously across the full trial. The SEM is
  small only because n is large, not because the group is homogeneous.
- **Visualize without presupposing groups first.** A single sorted heatmap or a
  peak-time histogram reveals the continuum that the three-panel presentation hides.

</div>
